## 1. Dependency

In [1]:
import csv
import yaml

## 2. Konfigurasi

In [ ]:
# dataset = "1sample"

CSV_INPUT = f"results/{dataset}/result-2-log-decoded.csv"
CSV_OUTPUT = f"results/{dataset}/result-3-low-level-ground-truth.csv"

# Path ke file YAML labelling rules
RULES_FILE = f"rules-ground-truth/{dataset}.yaml"

In [3]:
def count_lines(path):
    # fast line count
    cnt = 0
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for _ in f:
            cnt += 1
    return cnt

## 3. Load Labelling Rules dari YAML

Format YAML:
```yaml
- id: rule_name
  ground_truth_label: label_value
  sensitivity: moderate  # or 'strict'
  filter:
    - pattern1
    - pattern2  # AND condition
```

**Sensitivity Levels:**
- **moderate** (default): Case-insensitive, patterns can appear anywhere in text
- **strict**: Case-sensitive, respects `%` wildcards:
  - `%pattern%` = pattern can appear anywhere
  - `pattern%` = pattern must be at start
  - `%pattern` = pattern must be at end
  - `pattern` = exact match (no wildcards)

In [4]:
def load_rules_from_yaml(yaml_path):
    """
    Load labelling rules dari file YAML.
    
    Args:
        yaml_path: Path ke file YAML
    
    Returns:
        List of rule dictionaries
    """
    with open(yaml_path, 'r', encoding='utf-8') as f:
        rules = yaml.safe_load(f)
    
    print(f"Loaded {len(rules)} rules from {yaml_path}")
    return rules


# Load rules
labelling_rules = load_rules_from_yaml(RULES_FILE)

Loaded 39 rules from rules-ground-truth/organization-x.yaml


In [5]:
# Preview beberapa rules
for rule in labelling_rules[:3]:
    print(f"Rule: {rule['id']}")
    print(f"  Ground Truth: {rule['ground_truth_label']}")
    print(f"  Filters:      {rule['filter']}")
    print()

Rule: dir_scan
  Ground Truth: dir_scan
  Filters:      ['code: 404', 'Mozilla/4.0 (compatible; MSIE 8.0; Windows NT 5.1; Trident/4.0)']

Rule: dir_scan
  Ground Truth: dir_scan_go
  Filters:      ['Go-http-client']

Rule: dir_scan
  Ground Truth: dir_scan_python
  Filters:      ['code: 302', 'python']



## 4. Fungsi Pattern Matching

In [6]:
def match_pattern_strict(text, pattern):
    """
    Match pattern dengan strict mode (case-sensitive, respects % wildcards).
    
    Args:
        text: String yang akan dicek (case-sensitive)
        pattern: Pattern dengan % wildcards
    
    Returns:
        True jika match, False jika tidak
    
    Examples:
        '%Mozilla/5.0%' matches 'foo Mozilla/5.0 bar' (anywhere)
        'Mozilla/5.0%' matches 'Mozilla/5.0 bar' (start)
        '%Mozilla/5.0' matches 'foo Mozilla/5.0' (end)
        'Mozilla/5.0' matches 'Mozilla/5.0' (exact)
    """
    # Check wildcard positions
    starts_with_wildcard = pattern.startswith('%')
    ends_with_wildcard = pattern.endswith('%')
    
    # Remove wildcards
    clean_pattern = pattern.strip('%')
    
    # Apply matching logic based on wildcards
    if starts_with_wildcard and ends_with_wildcard:
        # %pattern% - can appear anywhere
        return clean_pattern in text
    elif starts_with_wildcard:
        # %pattern - must be at end
        return text.endswith(clean_pattern)
    elif ends_with_wildcard:
        # pattern% - must be at start
        return text.startswith(clean_pattern)
    else:
        # pattern - exact match
        return clean_pattern == text


def match_pattern_moderate(text, pattern):
    """
    Match pattern dengan moderate mode (case-insensitive, anywhere).
    
    Args:
        text: String yang akan dicek
        pattern: Pattern to match
    
    Returns:
        True jika match, False jika tidak
    """
    text_lower = text.lower()
    pattern_lower = str(pattern).lower()
    return pattern_lower in text_lower


def match_filters(text, filters, sensitivity='moderate'):
    """
    Match semua filter patterns terhadap text (AND condition).
    
    Args:
        text: String yang akan dicek
        filters: List of filter patterns
        sensitivity: 'moderate' (default) atau 'strict'
    
    Returns:
        True jika semua filter match, False jika tidak
    """
    for pattern in filters:
        pattern_str = str(pattern)
        
        if sensitivity == 'strict':
            # Strict mode: case-sensitive, respect % wildcards
            if not match_pattern_strict(text, pattern_str):
                return False
        else:
            # Moderate mode: case-insensitive, anywhere
            if not match_pattern_moderate(text, pattern_str):
                return False
    
    return True


def get_labels(text, rules):
    """
    Dapatkan labels berdasarkan text dan rules.
    
    Args:
        text: String yang akan dicek
        rules: List of rule dictionaries dari YAML
    
    Returns:
        ground_truth_label
    """
    for rule in rules:
        filters = rule.get('filter', [])
        sensitivity = rule.get('sensitivity', 'moderate')  # Default: moderate
        
        if match_filters(text, filters, sensitivity):
            return rule['ground_truth_label']
    
    # Default jika tidak ada yang match
    return 'benign'

## 4.5. Testing Sensitivity Modes

In [7]:
# Test cases untuk sensitivity modes
print("="*80)
print("TESTING MODERATE MODE (case-insensitive, anywhere)")
print("="*80)

test_text_moderate = "Accepted password for root from 27.116.61.59"
test_filters_moderate = ["Accepted password for", "27.116.61.59"]

result = match_filters(test_text_moderate, test_filters_moderate, 'moderate')
print(f"Text: {test_text_moderate}")
print(f"Filters: {test_filters_moderate}")
print(f"Result: {result}")
print()

# Test case-insensitive
test_text_case = "ACCEPTED PASSWORD FOR ROOT from 27.116.61.59"
result_case = match_filters(test_text_case, test_filters_moderate, 'moderate')
print(f"Text (uppercase): {test_text_case}")
print(f"Result: {result_case} (should be True - case insensitive)")
print()

print("="*80)
print("TESTING STRICT MODE (case-sensitive, % wildcards)")
print("="*80)

# Test 1: %pattern% (anywhere)
test_text_1 = "foo Mozilla/5.0 bar POST /wp-login.php 200"
test_filters_1 = ["%Mozilla/5.0%", "%/wp-login.php%", "%POST%", "%200%"]
result_1 = match_filters(test_text_1, test_filters_1, 'strict')
print(f"Test 1 - Anywhere match:")
print(f"  Text: {test_text_1}")
print(f"  Filters: {test_filters_1}")
print(f"  Result: {result_1} (should be True)")
print()

# Test 2: pattern% (starts with)
test_text_2 = "POST /wp-login.php"
test_filters_2 = ["POST%"]
result_2 = match_filters(test_text_2, test_filters_2, 'strict')
print(f"Test 2 - Starts with:")
print(f"  Text: {test_text_2}")
print(f"  Filters: {test_filters_2}")
print(f"  Result: {result_2} (should be True)")
print()

# Test 3: %pattern (ends with)
test_text_3 = "foo bar 200"
test_filters_3 = ["%200"]
result_3 = match_filters(test_text_3, test_filters_3, 'strict')
print(f"Test 3 - Ends with:")
print(f"  Text: {test_text_3}")
print(f"  Filters: {test_filters_3}")
print(f"  Result: {result_3} (should be True)")
print()

# Test 4: Case sensitivity
test_text_4 = "foo mozilla/5.0 bar"  # lowercase
test_filters_4 = ["%Mozilla/5.0%"]  # uppercase M
result_4 = match_filters(test_text_4, test_filters_4, 'strict')
print(f"Test 4 - Case sensitivity:")
print(f"  Text: {test_text_4}")
print(f"  Filters: {test_filters_4}")
print(f"  Result: {result_4} (should be False - case sensitive)")
print()

# Test 5: Exact match (no wildcards)
test_text_5 = "200"
test_filters_5 = ["200"]
result_5 = match_filters(test_text_5, test_filters_5, 'strict')
print(f"Test 5 - Exact match:")
print(f"  Text: {test_text_5}")
print(f"  Filters: {test_filters_5}")
print(f"  Result: {result_5} (should be True)")
print()

test_text_5b = "foo 200 bar"
result_5b = match_filters(test_text_5b, test_filters_5, 'strict')
print(f"Test 5b - Exact match (should fail):")
print(f"  Text: {test_text_5b}")
print(f"  Filters: {test_filters_5}")
print(f"  Result: {result_5b} (should be False - not exact match)")

TESTING MODERATE MODE (case-insensitive, anywhere)
Text: Accepted password for root from 27.116.61.59
Filters: ['Accepted password for', '27.116.61.59']
Result: True

Text (uppercase): ACCEPTED PASSWORD FOR ROOT from 27.116.61.59
Result: True (should be True - case insensitive)

TESTING STRICT MODE (case-sensitive, % wildcards)
Test 1 - Anywhere match:
  Text: foo Mozilla/5.0 bar POST /wp-login.php 200
  Filters: ['%Mozilla/5.0%', '%/wp-login.php%', '%POST%', '%200%']
  Result: True (should be True)

Test 2 - Starts with:
  Text: POST /wp-login.php
  Filters: ['POST%']
  Result: True (should be True)

Test 3 - Ends with:
  Text: foo bar 200
  Filters: ['%200']
  Result: True (should be True)

Test 4 - Case sensitivity:
  Text: foo mozilla/5.0 bar
  Filters: ['%Mozilla/5.0%']
  Result: False (should be False - case sensitive)

Test 5 - Exact match:
  Text: 200
  Filters: ['200']
  Result: True (should be True)

Test 5b - Exact match (should fail):
  Text: foo 200 bar
  Filters: ['200']


## 5. Eksekusi Labelling

In [8]:
total_lines = count_lines(CSV_INPUT)
print(f"Total lines in {CSV_INPUT}: {total_lines}")

processed = 0
labeled_count = 0

with open(CSV_INPUT, newline='', encoding="utf-8", errors='replace') as f, \
     open(CSV_OUTPUT, "w", newline='', encoding="utf-8") as out:
    
    reader = csv.DictReader(f)
    
    # Tambahkan kolom baru
    fieldnames = reader.fieldnames + ["ground_truth_label"]
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()
    
    for row in reader:
        processed += 1

        # Ambil kolom decoded untuk matching
        decoded = row.get("decoded", "")
        
        # Dapatkan labels berdasarkan pattern matching
        ground_truth_label = get_labels(decoded, labelling_rules)
        
        row["ground_truth_label"] = ground_truth_label
        
        if ground_truth_label != 'benign':
            labeled_count += 1

        writer.writerow(row)
            
        # Progress tiap 50.000 baris
        if processed % 50000 == 0:
            print(f"Processed {processed}/{total_lines} lines ({processed/total_lines:.2%})")

print(f"\nLabelling finished: {processed}/{total_lines} lines processed.")
print(f"Total labeled (non-benign): {labeled_count}")
print(f"Total benign: {processed - labeled_count}")

Total lines in results/organization-x/result-2-log-decoded.csv: 216761
Processed 50000/216761 lines (23.07%)
Processed 100000/216761 lines (46.13%)
Processed 150000/216761 lines (69.20%)
Processed 200000/216761 lines (92.27%)

Labelling finished: 216754/216761 lines processed.
Total labeled (non-benign): 672
Total benign: 216082
